In [ ]:
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost.sklearn import XGBClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn import metrics
from scipy.stats import zscore
from src.utils.data import getDataPandas

In [ ]:
data = getDataPandas()
data = data.sample(frac=1, random_state=1)
data = data.drop_duplicates(subset=['PATNO', 'EVENT_ID'], keep='first').reset_index(drop=True)

In [ ]:
data = data.drop(data[data['NUPDR3OF'] < 5].index).reset_index(drop=True)

In [ ]:
def load_img(rec):
    img_data = np.array(nib.load(rec.T1_MNI_PATH).get_fdata())
    return img_data

In [ ]:
data['T1'] = data.apply(load_img, axis=1)

In [ ]:
vox = np.array([np.array(l) for l in data['T1']])
vox = np.reshape(vox, (data.shape[0], -1))
vox = zscore(vox, axis=1)

In [ ]:
x = data[['NUPDR3ON', 'AGE_AT_VISIT', 'SEX', 'DURATION']]
y = data[['CAT']]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=12, stratify=y)

In [ ]:
print(len(y_train[y_train['CAT']==1]))
print(len(y_train[y_train['CAT']==0]))
print(len(y_test[y_test['CAT']==1]))
print(len(y_test[y_test['CAT']==0]))

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.9)
train_features = pca.fit_transform(vox[x_train.index])
train_fe = pd.DataFrame(train_features)
test_features = pca.transform(vox[x_test.index])
test_fe = pd.DataFrame(test_features)
x_train = x_train.reset_index(drop=True)
x_train = x_train.join(train_fe)
#x_train[['NUPDR3ON', 'AGE_AT_VISIT', 'DURATION']] = x_train[['NUPDR3ON', 'AGE_AT_VISIT', 'DURATION']].apply(zscore)
x_test = x_test.reset_index(drop=True)
x_test = x_test.join(test_fe)
#x_test[['NUPDR3ON', 'AGE_AT_VISIT', 'DURATION']] = x_test[['NUPDR3ON', 'AGE_AT_VISIT', 'DURATION']].apply(zscore)

In [ ]:
from scipy.stats import ttest_ind, chi2_contingency, normaltest, ranksums
print(normaltest(x_train['NUPDR3ON']))
print(normaltest(x_train['AGE_AT_VISIT']))
print(normaltest(x_train['DURATION']))
print()
print(normaltest(x_test['NUPDR3ON']))
print(normaltest(x_test['AGE_AT_VISIT']))
print(normaltest(x_test['DURATION']))
print()
print(ranksums(x_train['NUPDR3ON'], x_test['NUPDR3ON']))
print(ttest_ind(x_train['AGE_AT_VISIT'], x_test['AGE_AT_VISIT']))
print(ranksums(x_train['DURATION'], x_test['DURATION']))
_, p, _, _ = chi2_contingency([[len(x_test[x_test['SEX']==0]), len(x_train[x_train['SEX']==0])], [len(x_test[x_test['SEX']==1]), len(x_train[x_train['SEX']==1])]])
print(p)
_, p, _, _ = chi2_contingency([[len(y_test[y_test['CAT']==0]), len(y_train[y_train['CAT']==0])], [len(y_test[y_test['CAT']==1]), len(y_train[y_train['CAT']==1])]])
print(p)

In [ ]:
x1 = x[y['CAT']==1]
x0 = x[y['CAT']==0]
print(normaltest(x1['NUPDR3ON']))
print(normaltest(x1['AGE_AT_VISIT']))
print(normaltest(x1['DURATION']))
print()
print(normaltest(x0['NUPDR3ON']))
print(normaltest(x0['AGE_AT_VISIT']))
print(normaltest(x0['DURATION']))
print()
print(ranksums(x0['NUPDR3ON'], x1['NUPDR3ON']))
print(ranksums(x0['AGE_AT_VISIT'], x1['AGE_AT_VISIT']))
print(ranksums(x0['DURATION'], x1['DURATION']))
_, p, _, _ = chi2_contingency([[len(x1[x1['SEX']==0]), len(x0[x0['SEX']==0])], [len(x1[x1['SEX']==1]), len(x0[x0['SEX']==1])]])
print(p)

In [ ]:
model = XGBClassifier()
parameters = {'nthread': [4],
              'objective': ['binary:logistic'],
              'learning_rate': [0.2, 0.3],
              'max_depth': [1, 3, 5],
              'min_child_weight': [1, 3, 5],
              'subsample': [0.8, 0.85, 0.9],
              'colsample_bytree': [0.75, 0.8, 0.85],
              'n_estimators': [50, 100, 200],
              'missing': [-999],
              'random_state': [1]}
clf = GridSearchCV(model, parameters,
                        n_jobs=5,
                        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=1),
                        verbose=3,
                        #n_iter=200,
                        scoring='roc_auc')

In [ ]:
clf.fit(x_train, y_train)

In [ ]:
clf.best_params_

In [ ]:
y_prob = clf.best_estimator_.predict_proba(x_test)
print('AUC train {}, test {}'.format(clf.best_score_, metrics.roc_auc_score(list(y_test['CAT']), y_prob[:, 1])))

In [ ]:
y_pred = clf.best_estimator_.predict(x_train)
print(metrics.classification_report(list(y_train['CAT']), y_pred))
print(metrics.confusion_matrix(list(y_train['CAT']), y_pred))

In [ ]:
y_pred = clf.best_estimator_.predict(x_test)
print(metrics.classification_report(list(y_test['CAT']), y_pred))
print(metrics.confusion_matrix(list(y_test['CAT']), y_pred))